In [ ]:
import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="Corrs"
import xgboost as xgb
import umap
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# SHAP-set (XGBoost) — FAST refactor w/ PyTorch (+MPS) & notebook bars
# ============================================================
# pip install numpy pandas anndata scanpy xgboost scikit-learn scipy torch tqdm

from __future__ import annotations
import time
from typing import Dict, Set, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.calibration import calibration_curve
from scipy.stats import chi2
from xgboost import XGBClassifier
import xgboost as xgb
from tqdm.auto import tqdm

# ---------- optional torch (for GPU/MPS accel) ----------
try:
    import torch
    _TORCH_OK = True
except Exception:
    _TORCH_OK = False


# =======================
# Device / dtype helpers
# =======================
def _select_device(prefer_mps: bool = True) -> str:
    if not _TORCH_OK:
        return "cpu"
    if prefer_mps and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

_TORCH_DEVICE = _select_device(prefer_mps=True)
_TORCH_DTYPE = torch.float32   # IMPORTANT: MPS prefers float32


# =======================
# Utility helpers (API kept)
# =======================
def get_layer_matrix(adata, layer: str):
    """Return matrix from AnnData layer or X (float32)."""
    X = adata.layers[layer] if (layer and layer in adata.layers) else adata.X
    return np.asarray(X, dtype=np.float32)

def intersect_sets(var_names, marker_sets: Dict[str, Set[str]]):
    """Intersect sets with current feature space; return names, mask [S,G], sizes [S]."""
    var_names = np.asarray(var_names)
    names, masks, sizes = [], [], []
    for nm, genes in marker_sets.items():
        m = np.isin(var_names, list(genes))
        if m.any():
            names.append(nm)
            masks.append(m)
            sizes.append(int(m.sum()))
    if not names:
        raise ValueError("No marker/gene sets overlap the features (var_names).")
    Gmask = np.vstack(masks).astype(bool)
    return names, Gmask, np.asarray(sizes, int)

def bh_fdr(p):
    """Benjamini–Hochberg FDR (vectorized)."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    ranked = p[order]
    n = len(p)
    q = ranked * n / (np.arange(1, n + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    return out


# =======================
# XGBoost contribs helpers (TreeSHAP)
# =======================
def _predict_contribs_compat(clf: XGBClassifier, X_block: np.ndarray,
                             feature_names=None, approx: bool = False) -> np.ndarray:
    """
    Robustly fetch SHAP (TreeSHAP) contributions from XGBoost, with bias column.
    Works across older/newer xgboost versions.
    """
    # Try sklearn wrapper (newer xgboost supports pred_contribs on sklearn API)
    try:
        return clf.predict(X_block, pred_contribs=True)  # may raise TypeError
    except TypeError:
        pass
    # Fallback to Booster API (most robust)
    dm = xgb.DMatrix(X_block, feature_names=(list(feature_names) if feature_names is not None else None))
    return clf.get_booster().predict(
        dm,
        pred_contribs=True,          # exact TreeSHAP on margins
        approx_contribs=approx,
        validate_features=True
    )

def _xgb_contribs_to_tensor(pred_contribs: np.ndarray, expected_n_features: int | None = None) -> np.ndarray:
    """
    Convert XGBoost pred_contribs output to (N, C, G) without bias.
    If expected_n_features is given, trim/pad to match.
    """
    arr = np.asarray(pred_contribs, dtype=np.float32)
    if arr.ndim != 3:
        raise ValueError(f"Unexpected pred_contribs shape: {arr.shape}")
    A, B, C = arr.shape
    # Case A: (N, G+1, C)  -> drop bias, transpose -> (N, C, G)
    if C <= 64 and B >= 2:
        feats = B - 1
        out = np.transpose(arr[:, :feats, :], (0, 2, 1))  # (N, C, G)
    # Case B: (N, C, G+1)  -> drop bias -> (N, C, G)
    elif B <= 64 and C >= 2:
        feats = C - 1
        out = arr[:, :, :feats]
    else:
        raise ValueError(f"Can't interpret pred_contribs shape: {arr.shape}")

    if expected_n_features is not None and out.shape[2] != expected_n_features:
        Gc = out.shape[2]
        Ge = int(expected_n_features)
        if Gc > Ge:
            out = out[:, :, :Ge]  # trim
        elif Gc < Ge:
            pad = np.zeros((out.shape[0], out.shape[1], Ge - Gc), dtype=out.dtype)
            out = np.concatenate([out, pad], axis=2)
    return out


# =======================
# Torch-accelerated set scoring (MPS/CUDA/CPU)
# =======================
def _scores_for_sets_torch(
    SH_np: np.ndarray,     # (N, C, G), float32
    Gmask_np: np.ndarray,  # (S, G), bool
    sizes_np: np.ndarray,  # (S,)
    n_perm: int,
    signed_weights: np.ndarray | None = None,  # (S, G) float32 or None
    abs_variant: bool = True,
    seed: int | None = None,
    device: str = _TORCH_DEVICE,
) -> Tuple[np.ndarray, ...]:
    """
    Returns: set_mean, set_frac, set_rank, set_z, set_p, set_z_abs, set_p_abs
    Each shape (N, C, S). Abs arrays may be None if abs_variant=False.
    """
    if not _TORCH_OK:
        raise RuntimeError("PyTorch is required for the accelerated path.")

    torch.manual_seed(seed or 0)
    N, C, G = SH_np.shape
    S = Gmask_np.shape[0]

    SH = torch.from_numpy(SH_np).to(device=device, dtype=_TORCH_DTYPE)          # [N,C,G]
    Gmask = torch.from_numpy(Gmask_np).to(device=device)                         # [S,G] bool
    sizes = torch.from_numpy(sizes_np.astype(np.int32)).to(device=device)        # [S]
    SW = None
    if signed_weights is not None:
        SW = torch.from_numpy(signed_weights.astype(np.float32)).to(device=device)

    # (A) set sum / mean
    if SW is not None:
        set_sum_default = torch.einsum("ncg,sg->ncs", SH, Gmask.float())
        set_sum_signed = torch.einsum("ncg,sg->ncs", SH, SW)
        use_signed = (SW.abs().sum(dim=1) > 0).view(1, 1, S)  # [1,1,S]
        set_sum = torch.where(use_signed, set_sum_signed, set_sum_default)
    else:
        set_sum = torch.einsum("ncg,sg->ncs", SH, Gmask.float())

    set_mean = set_sum / torch.clamp(sizes.view(1, 1, S).float(), min=1.0)

    # (B) signed fraction
    denom = SH.abs().sum(dim=2, keepdim=True) + 1e-12
    set_frac = (set_sum / denom).to(_TORCH_DTYPE)

    # (C) rank-U
    # order = torch.argsort(-SH, dim=2)                                            # [N,C,G]
    # ranks = torch.empty_like(order)
    # arange_g = torch.arange(G, device=device).view(1, 1, -1)
    # ranks.scatter_(2, order, arange_g)
    # ranks = ranks + 1

    # (C) rank-U (inverse permutation via argsort-of-argsort; avoids scatter_ broadcasting issues)
    order = torch.argsort(-SH, dim=2)            # [N,C,G], each row is permutation of 0..G-1
    ranks = torch.argsort(order, dim=2) + 1      # [N,C,G], ranks in 1..G

    
    set_rank = torch.empty((N, C, S), device=device, dtype=_TORCH_DTYPE)
    m_const = (sizes * (sizes + 1) / 2.0).to(_TORCH_DTYPE)
    for s in range(S):
        m = int(sizes[s].item())
        if m == 0:
            set_rank[:, :, s] = 0
            continue
        idx = torch.nonzero(Gmask[s], as_tuple=False).view(-1)
        r = torch.index_select(ranks, 2, idx)
        U = r.sum(dim=2) - m_const[s]
        if m == G:
            set_rank[:, :, s] = 1.0
        else:
            set_rank[:, :, s] = (U / (m * (G - m))).to(_TORCH_DTYPE)

    # (D) permutation nulls (shared per unique m)
    uniq_m = torch.unique(sizes).tolist()
    set_z = torch.empty((N, C, S), device=device, dtype=_TORCH_DTYPE)
    set_p = torch.empty((N, C, S), device=device, dtype=torch.float32)
    set_z_abs = set_p_abs = None
    if abs_variant:
        set_z_abs = torch.empty_like(set_z)
        set_p_abs = torch.empty_like(set_p)

    SH_flat = SH.reshape(N * C, G)
    abs_SH_flat = SH_flat.abs() if abs_variant else None

    for m in uniq_m:
        m = int(m)
        if m <= 0:
            continue
        idxs = torch.stack([torch.randperm(G, device=device)[:m] for _ in range(n_perm)], dim=0)  # [P,m]

        base = SH_flat.unsqueeze(1).expand(-1, n_perm, -1)
        gathered = torch.gather(base, 2, idxs.unsqueeze(0).expand(N*C, -1, -1))
        null = gathered.sum(dim=2)                                               # [NC,P]
        mu = null.mean(dim=1, keepdim=True)
        sd = null.std(dim=1, unbiased=False, keepdim=True) + 1e-6

        if abs_variant:
            abase = abs_SH_flat.unsqueeze(1).expand(-1, n_perm, -1)
            agather = torch.gather(abase, 2, idxs.unsqueeze(0).expand(N*C, -1, -1))
            anull = agather.sum(dim=2)
            amu = anull.mean(dim=1, keepdim=True)
            asd = anull.std(dim=1, unbiased=False, keepdim=True) + 1e-6

        sets_of_m = torch.nonzero(sizes == m, as_tuple=False).view(-1)
        if sets_of_m.numel() == 0:
            continue

        for s in sets_of_m.tolist():
            obs = set_sum[:, :, s].reshape(N * C, 1)
            z = ((obs - mu) / sd).reshape(N, C)
            ge = (null >= obs).sum(dim=1).reshape(N, C)
            p = (ge + 1.0) / (n_perm + 1.0)
            set_z[:, :, s] = z.to(_TORCH_DTYPE)
            set_p[:, :, s] = p.to(torch.float32)

            if abs_variant:
                aobs = (set_sum[:, :, s].abs()).reshape(N * C, 1)
                az = ((aobs - amu) / asd).reshape(N, C)
                age = (anull >= aobs).sum(dim=1).reshape(N, C)
                ap = (age + 1.0) / (n_perm + 1.0)
                set_z_abs[:, :, s] = az.to(_TORCH_DTYPE)
                set_p_abs[:, :, s] = ap.to(torch.float32)

    def _to_np(x):
        return x.detach().to("cpu").numpy()

    out = (
        _to_np(set_mean),
        _to_np(set_frac),
        _to_np(set_rank),
        _to_np(set_z),
        _to_np(set_p),
        _to_np(set_z_abs) if abs_variant else None,
        _to_np(set_p_abs) if abs_variant else None,
    )
    # free memory
    del SH, Gmask, sizes, order, ranks
    return out


# =======================
# OOF metrics (same logic)
# =======================
def oof_classifier_metrics_xgb(X, y_int, n_splits=5, seed=0, model_kwargs=None):
    from sklearn.metrics import log_loss, confusion_matrix
    model_kwargs = model_kwargs or {}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    base = XGBClassifier(
        objective="multi:softprob",
        n_estimators=600,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=0,
        **model_kwargs,
    )
    C = len(np.unique(y_int))
    probs = np.zeros((len(y_int), C), dtype=np.float32)
    preds = np.empty(len(y_int), dtype=int)
    for tr, te in skf.split(X, y_int):
        clf = XGBClassifier(**base.get_params())
        clf.set_params(random_state=int(np.random.default_rng(seed).integers(1, 2**31 - 1)))
        clf.fit(X[tr], y_int[tr])
        P = clf.predict_proba(X[te])
        probs[te] = P
        preds[te] = np.argmax(P, axis=1)
    macro_f1 = f1_score(y_int, preds, average="macro")
    micro_f1 = f1_score(y_int, preds, average="micro")
    Y = pd.get_dummies(y_int)
    try:
        macro_auroc = roc_auc_score(Y, probs, average="macro", multi_class="ovr")
    except ValueError:
        macro_auroc = np.nan
    ll = log_loss(y_int, probs)
    brier = float(np.mean([((y_int == i).astype(int) - probs[:, i]) ** 2 for i in range(C)]))
    conf = probs.max(1); acc = (preds == y_int).astype(int)
    fracs, means = calibration_curve(acc, conf, n_bins=10, strategy="quantile")
    ece = float(np.mean(np.abs(fracs - means)))
    cm = pd.DataFrame(
        confusion_matrix(y_int, preds, labels=np.arange(C)),
        index=[f"true_{i}" for i in range(C)],
        columns=[f"pred_{i}" for i in range(C)],
    )
    return {
        "classes_int": np.arange(C),
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "macro_auroc": macro_auroc,
        "log_loss": ll,
        "brier": brier,
        "ECE": ece,
        "confusion_matrix": cm,
        "oof_probs": probs,
        "oof_preds": preds,
    }


# =======================
# Main FAST function (with Jupyter bars + robust alignment)
# =======================
def shap_cluster_gene_set_scores_xgb_fast(
    adata,
    marker_sets: Dict[str, Set[str]],
    layer: str = "arcsinh",
    cluster_key: str = "cluster",
    do_cluster: bool = False,
    leiden_res: float = 1.0,
    neighbors_k: int = 15,
    n_repeats: int = 2,
    n_splits: int = 5,
    n_perm: int = 200,
    random_state: int = 0,
    model_kwargs: dict | None = None,
    signed_weights: Dict[str, Dict[str, float]] | None = None,
    return_metrics: bool = True,
    class_agg: str = "prob",            # 'prob' | 'mean' | 'max' | 'pred'
    abs_variant: bool = True,           # compute abs(Z) permutation too
):
    """
    Fast SHAP-set with:
      - XGB TreeSHAP via pred_contribs (Booster fallback for compatibility)
      - Torch-accelerated set scoring (MPS/CUDA/CPU)
      - Defensive feature-axis alignment (avoids einsum size mismatch)
      - Jupyter-friendly progress bars
    """
    import scanpy as sc

    t0 = time.perf_counter()
    rng_master = np.random.default_rng(random_state)

    # Optional pseudo-label clustering
    if (cluster_key not in adata.obs) and do_cluster:
        rep = "X_cytovi" if "X_cytovi" in adata.obsm else None
        print(f"[info] Building neighbors (k={neighbors_k}) and Leiden (res={leiden_res})…")
        sc.pp.neighbors(adata, use_rep=rep, n_neighbors=neighbors_k)
        sc.tl.leiden(adata, key_added=cluster_key, resolution=leiden_res)

    # Encode labels
    y_str = adata.obs[cluster_key].astype(str).to_numpy()
    le = LabelEncoder().fit(y_str)
    y_int = le.transform(y_str)
    class_names = le.classes_
    C_master = len(class_names)

    # Feature matrix (float32)
    X_all = get_layer_matrix(adata, layer)
    genes_all = np.asarray(adata.var_names)

    # Restrict to union of set members (speed win)
    union = np.zeros(len(genes_all), bool)
    for S in marker_sets.values():
        union |= np.isin(genes_all, list(S))
    if union.sum() >= 5:
        X = X_all[:, union].astype(np.float32, copy=False)
        genes = genes_all[union]
    else:
        X = X_all.astype(np.float32, copy=False)
        genes = genes_all

    # Master sets (over full universe for stable column naming)
    set_names_master, _, _ = intersect_sets(genes_all, marker_sets)
    if len(set_names_master) == 0:
        raise ValueError("None of the marker sets overlap features.")
    S_master = len(set_names_master)
    name_to_master = {nm: i for i, nm in enumerate(set_names_master)}

    # CV base model
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    base = XGBClassifier(
        objective="multi:softprob",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=0,
        **(model_kwargs or {}),
    )

    # Optional signed weights over full universe
    signed_W_master = None
    if signed_weights is not None:
        signed_W_master = np.zeros((S_master, len(genes_all)), dtype=np.float32)
        for si, sname in enumerate(set_names_master):
            if sname in signed_weights:
                for feat, w in signed_weights[sname].items():
                    j = np.where(genes_all == feat)[0]
                    if j.size:
                        signed_W_master[si, j[0]] = np.float32(w)

    N, G = X.shape[0], X.shape[1]
    print(f"[start] device={_TORCH_DEVICE}  dtype=float32  N={N}  G={G}  C={C_master}  "
          f"repeats={n_repeats}  folds={n_splits}  perms={n_perm}")

    # Containers across repeats
    mean_runs, rank_runs, frac_runs, z_runs, p_runs = [], [], [], [], []
    z_abs_runs, p_abs_runs = [], []
    prob_runs, pred_runs = [], []

    with tqdm(total=n_repeats, desc="Repeats", position=0, leave=False, dynamic_ncols=True) as pbar_rep:
        for rep_idx in range(n_repeats):
            rng = np.random.default_rng(rng_master.integers(1, 2**31 - 1))

            mean_oof = np.zeros((N, C_master, S_master), np.float32)
            rank_oof = np.zeros((N, C_master, S_master), np.float32)
            frac_oof = np.zeros((N, C_master, S_master), np.float32)
            z_oof    = np.zeros((N, C_master, S_master), np.float32)
            p_oof    = np.ones((N, C_master, S_master),  np.float32)
            if abs_variant:
                z_oof_abs = np.zeros((N, C_master, S_master), np.float32)
                p_oof_abs = np.ones((N, C_master, S_master),  np.float32)

            oof_probs = np.zeros((N, C_master), np.float32)
            oof_pred  = np.zeros(N, dtype=int)

            with tqdm(total=n_splits, desc=f"CV folds (rep {rep_idx+1}/{n_repeats})",
                      position=1, leave=False, dynamic_ncols=True) as pbar_fold:

                for tr, te in skf.split(X, y_int):
                    clf = XGBClassifier(**base.get_params())
                    clf.set_params(random_state=int(rng.integers(1, 2**31 - 1)))
                    clf.fit(X[tr], y_int[tr])

                    # OOF probs/preds aligned to master class order
                    P = clf.predict_proba(X[te])  # (te, C_fold)
                    classes_fold = getattr(clf, "classes_", np.arange(P.shape[1]))
                    for i, c in enumerate(classes_fold):
                        c = int(c)
                        if 0 <= c < C_master:
                            oof_probs[te, c] = P[:, i]
                    oof_pred[te] = np.argmax(oof_probs[te], axis=1)

                    # ---- FAST SHAP via pred_contribs (TreeSHAP; margins) ----
                    contribs = _predict_contribs_compat(clf, X[te], feature_names=genes, approx=False)
                    SH = _xgb_contribs_to_tensor(contribs, expected_n_features=genes.shape[0])  # (te, C_fold, G_fold)

                    # ---- Defensive alignment of feature axis ----
                    feats_axis = genes                    # intended order used in training
                    G_from_shap = SH.shape[2]

                    if feats_axis.shape[0] != G_from_shap:
                        # enforce same length as SHAP contributions
                        if feats_axis.shape[0] > G_from_shap:
                            feats_axis_eff = feats_axis[:G_from_shap]
                        else:
                            Gmin = min(G_from_shap, feats_axis.shape[0])
                            feats_axis_eff = feats_axis[:Gmin]
                            SH = SH[:, :, :Gmin]
                    else:
                        feats_axis_eff = feats_axis

                    # Build mask & sizes on the effective feature list
                    fold_set_names, Gmask_fold, sizes_fold = intersect_sets(feats_axis_eff, marker_sets)
                    if len(fold_set_names) == 0:
                        pbar_fold.update(1)
                        continue
                    fold_to_master = np.array([name_to_master[nm] for nm in fold_set_names], int)

                    # Project optional signed weights to this axis
                    SW_fold = None
                    if signed_W_master is not None:
                        # map feats_axis_eff -> genes_all positions
                        # fast path: try searchsorted (requires sorted genes_all)
                        try:
                            pos_in_all = np.searchsorted(genes_all, feats_axis_eff)
                            if not np.all(genes_all[pos_in_all] == feats_axis_eff):
                                raise ValueError
                        except Exception:
                            pos_map = {g: i for i, g in enumerate(genes_all)}
                            pos_in_all = np.array([pos_map[g] for g in feats_axis_eff], int)
                        SW_fold = signed_W_master[fold_to_master][:, pos_in_all]  # [S_fold, G_eff]

                    # ---- Torch-accelerated scoring (with permutations) ----
                    set_mean, set_frac, set_rank, set_z, set_p, set_z_abs, set_p_abs = _scores_for_sets_torch(
                        SH_np=SH,
                        Gmask_np=Gmask_fold,
                        sizes_np=sizes_fold,
                        n_perm=n_perm,
                        signed_weights=SW_fold,
                        abs_variant=abs_variant,
                        seed=int(rng.integers(1, 2**31 - 1)),
                        device=_TORCH_DEVICE,
                    )

                    # Map to master class slots
                    C_fold = SH.shape[1]
                    if len(classes_fold) != C_fold:
                        classes_fold = np.arange(C_fold)
                    class_pos_sh = np.full(C_fold, -1, dtype=int)
                    for i, c in enumerate(classes_fold):
                        c = int(c)
                        if 0 <= c < C_master:
                            class_pos_sh[i] = c

                    for i_c, cpos in enumerate(class_pos_sh):
                        if cpos < 0:
                            continue
                        idx = np.ix_(te, [cpos], fold_to_master)
                        mean_oof[idx] = set_mean[:, i_c, :][:, None, :]
                        rank_oof[idx] = set_rank[:, i_c, :][:, None, :]
                        frac_oof[idx] = set_frac[:, i_c, :][:, None, :]
                        z_oof[idx]    = set_z[:, i_c, :][:, None, :]
                        p_oof[idx]    = set_p[:, i_c, :][:, None, :]
                        if abs_variant:
                            z_oof_abs[idx] = set_z_abs[:, i_c, :][:, None, :]
                            p_oof_abs[idx] = set_p_abs[:, i_c, :][:, None, :]

                    pbar_fold.update(1)

            mean_runs.append(mean_oof); rank_runs.append(rank_oof); frac_runs.append(frac_oof)
            z_runs.append(z_oof); p_runs.append(p_oof)
            if abs_variant:
                z_abs_runs.append(z_oof_abs); p_abs_runs.append(p_oof_abs)
            prob_runs.append(oof_probs); pred_runs.append(oof_pred)

            pbar_rep.update(1)

    # ===== Combine repeats =====
    mean_arr = np.stack(mean_runs, 0).mean(0)                 # [N,C,S]
    rank_arr = np.stack(rank_runs, 0).mean(0)
    frac_arr = np.stack(frac_runs, 0).mean(0)
    z_arr    = np.stack(z_runs, 0).mean(0)
    p_arr    = np.stack(p_runs, 0)                            # [R,N,C,S]

    stat = -2.0 * np.sum(np.log(np.clip(p_arr, 1e-300, 1.0)), axis=0)
    p_comb = 1.0 - chi2.cdf(stat, 2 * p_arr.shape[0])        # [N,C,S]
    fdr_comb = bh_fdr(p_comb.ravel()).reshape(p_comb.shape)

    if abs_variant:
        z_abs_arr = np.stack(z_abs_runs, 0).mean(0)
        p_abs_arr = np.stack(p_abs_runs, 0)
        stat_abs = -2.0 * np.sum(np.log(np.clip(p_abs_arr, 1e-300, 1.0)), axis=0)
        p_comb_abs = 1.0 - chi2.cdf(stat_abs, 2 * p_abs_arr.shape[0])
        fdr_comb_abs = bh_fdr(p_comb_abs.ravel()).reshape(p_comb_abs.shape)

    probs_mean = np.stack(prob_runs, 0).mean(0)               # [N,C]
    pred_mode  = np.round(np.stack(pred_runs, 0).mean(0)).astype(int)

    def agg_class(T: np.ndarray) -> np.ndarray:
        if T is None:
            return None
        if class_agg == "prob":
            Wc = probs_mean / (probs_mean.sum(1, keepdims=True) + 1e-12)
            return (T * Wc[:, :, None]).sum(1)
        elif class_agg == "mean":
            return T.mean(1)
        elif class_agg == "max":
            return T.max(1)
        elif class_agg == "pred":
            rows = np.arange(T.shape[0])
            return T[rows, pred_mode, :]
        else:
            raise ValueError("class_agg must be one of {'prob','mean','max','pred'}")

    mean_cell = agg_class(mean_arr)
    rank_cell = agg_class(rank_arr)
    frac_cell = agg_class(frac_arr)
    z_cell    = agg_class(z_arr)
    p_cell    = agg_class(p_comb)
    fdr_cell  = agg_class(fdr_comb)

    # Save to AnnData (cells × sets)
    adata.obsm["shapset_mean"]        = pd.DataFrame(mean_cell, index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_rankucell"]   = pd.DataFrame(rank_cell, index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_frac"]        = pd.DataFrame(frac_cell, index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_z"]           = pd.DataFrame(z_cell,    index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_pval"]        = pd.DataFrame(p_cell,    index=adata.obs_names, columns=set_names_master)
    adata.obsm["shapset_fdr"]         = pd.DataFrame(fdr_cell,  index=adata.obs_names, columns=set_names_master)

    # Per-class matrices (kept)
    for j, cls_name in enumerate(class_names):
        adata.obsm[f"shapset_mean_{cls_name}"]      = pd.DataFrame(mean_arr[:, j, :], index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_rankucell_{cls_name}"] = pd.DataFrame(rank_arr[:, j, :], index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_frac_{cls_name}"]      = pd.DataFrame(frac_arr[:, j, :], index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_z_{cls_name}"]         = pd.DataFrame(z_arr[:, j, :],   index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_pval_{cls_name}"]      = pd.DataFrame(p_comb[:, j, :],  index=adata.obs_names, columns=set_names_master)
        adata.obsm[f"shapset_fdr_{cls_name}"]       = pd.DataFrame(fdr_comb[:, j, :],index=adata.obs_names, columns=set_names_master)

    if abs_variant:
        z_abs_cell   = agg_class(z_abs_arr)
        p_abs_cell   = agg_class(p_comb_abs)
        fdr_abs_cell = agg_class(fdr_comb_abs)
        adata.obsm["shapset_z_abs"]    = pd.DataFrame(z_abs_cell,   index=adata.obs_names, columns=set_names_master)
        adata.obsm["shapset_pval_abs"] = pd.DataFrame(p_abs_cell,   index=adata.obs_names, columns=set_names_master)
        adata.obsm["shapset_fdr_abs"]  = pd.DataFrame(fdr_abs_cell, index=adata.obs_names, columns=set_names_master)

    clf_metrics = None
    if return_metrics:
        clf_metrics = oof_classifier_metrics_xgb(
            X, y_int, n_splits=n_splits, seed=random_state, model_kwargs=model_kwargs
        )

    dt = time.perf_counter() - t0
    print(f"[done] SHAP-set complete in {dt:.1f}s (device={_TORCH_DEVICE})")

    out = {
        "classes": class_names,
        "set_names": set_names_master,
        "scores_mean": mean_arr, "scores_rank": rank_arr,
        "scores_frac": frac_arr, "scores_z": z_arr,
        "pvals_cell_classwise": p_comb, "fdr_cell_classwise": fdr_comb,
        "scores_mean_cell": mean_cell, "scores_rank_cell": rank_cell,
        "scores_frac_cell": frac_cell, "scores_z_cell": z_cell,
        "pvals_cell": p_cell, "fdr_cell": fdr_cell,
        "class_agg": class_agg,
        "device": _TORCH_DEVICE,
        "elapsed_sec": dt,
    }
    if abs_variant:
        out.update({
            "scores_z_abs_cell": adata.obsm["shapset_z_abs"].to_numpy(),
            "pvals_abs_cell":    adata.obsm["shapset_pval_abs"].to_numpy(),
            "fdr_abs_cell":      adata.obsm["shapset_fdr_abs"].to_numpy(),
        })
    if clf_metrics is not None:
        out["clf_metrics"] = clf_metrics
    return out


In [ ]:
import glob

In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/"

In [ ]:
FList=glob.glob(dir+"norm*")
FList.sort()
FList

In [ ]:
FList=[
    '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_11.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_13.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_14.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_15.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_17.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_18.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_19.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_20.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_4.0.parquet',
# '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_4.1.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_5.0.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_7.0.parquet',
# '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_7.1.parquet',
 '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_8.0.parquet',
# '/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_8.1.parquet'
]

In [ ]:
Numbs=[11,13,14,15,17,18,19,20,4,5,7,8]

In [ ]:
Numbs.sort()
Numbs

In [ ]:
DBs=[f"BCK{N}" for N in Numbs]
DBs

In [ ]:
for DB,N in zip(DBs,Numbs):
    print(DB)
    globals()[DB]=pd.read_parquet(f"/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/normalized_not_scaled_{N}.0.parquet")
    print(DB,globals()[DB].shape)

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
Rep

In [ ]:
for DB in DBs:
#    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)
    try:
        globals()[DB].drop(columns=['DNA1','DNA2','Event #'],inplace=True)        
    except:
        pass




In [ ]:
N=list(globals()[DBs[0]].columns)
N.sort()
#N.remove('N-cadherin')
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)
NamesAll.remove("class")

In [ ]:
for DB in DBs:
    globals()[f"Label_{DB}"]=globals()[DB]["class"].values

In [ ]:
for DB in DBs:
    globals()[DB]=globals()[DB][NamesAll]
#    globals()[DB]['Samp']=DB

In [ ]:
Label_BCK11

In [ ]:
NC=2000
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC,replace=False)]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
#    globals()[DB]=(globals()[DB]-m)/s
#    globals()[DB]=np.arcsinh(globals()[DB]/5)
    globals()[DB]['Samp']=DB
    globals()[DB]['Class']=globals()[f"Label_{DB}"]

In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
DBs[0]

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs[1:2]:
    CAll=pd.concat([CAll,globals()[DB].sample(5000,replace=False)]).copy()

CAll=globals()[f"{DBs[1]}"].copy()
UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=None,verbose=True)

X_2d=UM.fit_transform(CAll[NamesAll])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
CAll['x']=X_2d[:,0]
CAll['y']=X_2d[:,1]

In [ ]:
CAll.reset_index(drop=True,inplace=True)

In [ ]:
AD=ad.AnnData(CAll[NamesAll],obs=CAll[['Samp','Class']])
AD.obsm['X_umap']=X_2d


In [ ]:
sc.pl.umap(AD,color=['Class','Samp'])

In [ ]:
%matplotlib inline

In [ ]:
CAll

In [ ]:
for DB in DBs:

    globals()[DB]['Samp']=DB
    print(DB,globals()[DB].shape[0])

In [ ]:
# ==== 1) Define gene/marker sets that match your panel names exactly ====
marker_sets = {
    # Epithelial & lineage
    "Epithelial_Luminal": {"EpCAM", "E-cadherin", "ER", "GATA3", "Pan-KRT", "KRT8-18"},
    "Basal_like": {"KRT5", "CD44", "Vimentin"},  # basal/intermediate cytokeratin + mesenchymal marker
    "Basal_Noa":{'H3K4me1', 'H3K9me2', 'H3K4me3'},
    # Stemness / tumor-initiating (mix of surface & chromatin)
#    "Stem_Prog": {"CD44", "CD24", "CD49f", "BMI1", "EZH2"},

    # EMT / mesenchymal programs (use signed weights below to penalize E-cadherin)
    "EMT": {"Vimentin", "aSMA", "CD44", "E-cadherin"},

    # Proliferation / cell cycle
    "Proliferation": {"KI67", "H3S28p", "H3K9ac", "H3K64ac"},

    # DNA damage / repair
#    "DNA_Damage": {"pH2A.X", "H2AK119ub"},

    # Polycomb repression (PRC1/2)
#    "Polycomb_Repression": {"H3K27me3", "H3K27me2", "EZH2", "H2AK119ub"},

    # Enhancer activation
#    "Active_Enhancer": {"H3K27ac", "H3K4me1", "H3K64ac", "H4K16ac", "H3K9ac"},

    # Transcriptional elongation / gene body
#    "Elongation_H3K36": {"H3K36me3", "H3K36me2"},

    # Promoter activation/repression
#    "Promoter_Active": {"H3K4me3", "H3K9ac", "H3K27ac"},
#    "Promoter_Poised_Bivalent": {"H3K4me3", "H3K27me3"},  # bivalency

    # Heterochromatin / compaction
#    "Heterochromatin": {"H3K9me3", "H3K9me2", "H4K20me3"},

    # Broad histone cores (optional coarse sets)
#    "Core_Histones": {"H3", "H4", "H3.3"},
}

# ==== 2) Signed weights for mixed-direction sets (optional) ====
# These let you encode patterns like CD44^hi / CD24^lo or EMT up with E-cadherin down.
signed_weights = {
    # CD44 high, CD24 low
    "Stem_Prog": {"CD44": +1.0, "CD24": -1.0},
    # EMT up, E-cad down
    "EMT": {"Vimentin": +1.0, "aSMA": +1.0, "CD44": +0.5, "E-cadherin": -1.0},
}

In [ ]:
import umap
import anndata as ad
import scanpy as sc

In [ ]:
%matplotlib inline

In [ ]:
MRK_All=N.copy()
MRK_All.remove('H3')
MRK_All.remove('H3.3')
MRK_All.remove('H4')


In [ ]:
import scanpy as sc
GS=list(marker_sets.keys())

In [ ]:
GS

In [ ]:
# ===========================
# Demo: permutation scorer on CyTOF (Python-only)
# ===========================
# pip install -q PyCytoData anndata scanpy numpy pandas scipy tqdm scikit-learn matplotlib

import os, itertools
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from math import comb
from typing import Dict, Set, Tuple, List, Optional
from tqdm.auto import tqdm
from scipy.stats import norm
from sklearn.metrics import roc_auc_score

# ---------------------------
# Data loading (PyCytoData)
# ---------------------------
try:
    from PyCytoData import DataLoader
except Exception as e:
    raise RuntimeError("Please `pip install PyCytoData` (or conda) before running this demo.") from e

def to_anndata_from_pycytodata(exprs) -> ad.AnnData:
    X = exprs.expression_matrix.astype(np.float32)
    var = pd.DataFrame(index=pd.Index(np.asarray(exprs.channels, dtype=str), name="marker"))
    obs = pd.DataFrame(index=pd.Index(np.arange(X.shape[0]).astype(str), name="cell"))
    if getattr(exprs, "cell_types", None) is not None:
        obs["cell_type"] = np.asarray(exprs.cell_types, dtype=object)
    if getattr(exprs, "sample_index", None) is not None:
        obs["sample"] = np.asarray(exprs.sample_index, dtype=object)
    adata = ad.AnnData(X=X, obs=obs, var=var)
    adata.var_names = var.index
    return adata

def load_benchmark(dataset: str, sample: Optional[List[str]] = None, preprocess: bool = True) -> ad.AnnData:
    exprs = DataLoader.load_dataset(dataset=dataset, sample=sample, preprocess=preprocess)
    adata = to_anndata_from_pycytodata(exprs)
    adata.layers["arcsinh"] = adata.X.copy()
    return adata

# ---------------------------
# Utility + marker sets
# ---------------------------
def get_layer_matrix(adata, layer: Optional[str]):
    return np.asarray(adata.layers[layer] if (layer and layer in adata.layers) else adata.X, dtype=np.float32)

def intersect_sets(var_names, marker_sets: Dict[str, Set[str]]):
    var_names = np.asarray(var_names)
    names, masks, sizes = [], [], []
    for nm, genes in marker_sets.items():
        m = np.isin(var_names, list(genes))
        if m.any():
            names.append(nm); masks.append(m); sizes.append(int(m.sum()))
    if not names: raise ValueError("No marker sets overlap the features.")
    return names, np.vstack(masks).astype(bool), np.asarray(sizes, int)

def robust_scale_markers(adata, layer_in="arcsinh", layer_out="scaled"):
    X = get_layer_matrix(adata, layer_in)
    med = np.median(X, axis=0, keepdims=True)
    mad = np.median(np.abs(X - med), axis=0, keepdims=True) + 1e-6
    Z = (X - med) / mad
    adata.layers[layer_out] = Z.astype(np.float32)
    return adata

def safe_intersect(names: np.ndarray, sets: Dict[str, Set[str]]) -> Dict[str, Set[str]]:
    nm_l = np.char.lower(np.asarray(names, dtype=str))
    out = {}
    for sname, genes in sets.items():
        want = set()
        for g in genes:
            g_low = g.lower()
            hit = (nm_l == g_low) | np.char.startswith(nm_l, g_low + "(") | np.char.startswith(nm_l, g_low + "-")
            if hit.any():
                want.update(names[hit])
        if want:
            out[sname] = want
    return out



# ---------------------------
# Your permutation scorer (v2 with Z_dir)
# ---------------------------
def sipsic_like_scores_v2(
    adata,
    marker_sets: Dict[str, Set[str]],
    layer: str = "scaled",
    n_perm: int = 2000,
    seed: int = 0,
    exclude_set: bool = True,
    two_sided: bool = False,
    abs_variant: bool = True,
    exact_max_combinations: int = 50_000,
    progress: bool = True,
):
    rng = np.random.default_rng(seed)
    X = get_layer_matrix(adata, layer)
    genes = np.asarray(adata.var_names)
    set_names, Gmask, sizes = intersect_sets(genes, marker_sets)
    N, G = X.shape
    S = len(set_names)
    # Row-centering
    Xc = X - X.mean(axis=1, keepdims=True)
    if abs_variant:
        Xc_abs = np.abs(Xc)
    # Observed sums
    obs = Xc @ Gmask.T
    if abs_variant:
        obs_abs = Xc_abs @ Gmask.T
    Z = np.zeros((N, S), np.float32); P = np.ones((N, S), np.float64)
    if abs_variant:
        Z_abs = np.zeros((N, S), np.float32); P_abs = np.ones((N, S), np.float64)

    def p_from_Z(z): return (2.0 * norm.sf(np.abs(z))) if two_sided else norm.sf(z)

    def update_stream(mu, m2, n_seen, batch):
        B = batch.shape[1]
        if B == 0: return mu, m2, n_seen
        bmean = batch.mean(axis=1); bm2 = ((batch - bmean[:, None])**2).sum(axis=1)
        new_n = n_seen + B; delta = bmean - mu
        new_mu = mu + delta * (B / new_n)
        new_m2 = m2 + bm2 + (delta**2) * n_seen * B / new_n
        return new_mu, new_m2, new_n

    it = range(S)
    if progress:
        it = tqdm(it, desc="Scoring sets (permutation/null)", leave=False)

    for s in it:
        m = int(sizes[s]); mask = Gmask[s]
        pool = np.where(~mask)[0] if exclude_set else np.arange(G)
        if exclude_set and len(pool) < m:  # fallback if complement too small
            pool = np.arange(G)
        do_enum = False; n_pool = len(pool); n_comb = float("inf")
        try:
            if 0 <= m <= n_pool and m <= 50:
                n_comb = comb(n_pool, m)
                if n_comb <= exact_max_combinations: do_enum = True
        except OverflowError:
            pass

        mu = np.zeros(N, np.float64); m2 = np.zeros(N, np.float64); n_seen = 0
        if abs_variant:
            mu_a = np.zeros(N, np.float64); m2_a = np.zeros(N, np.float64); n_seen_a = 0

        if do_enum:
            chunk = 4096
            itc = itertools.combinations(pool, m)
            while True:
                chunk_idx = list(itertools.islice(itc, chunk))
                if not chunk_idx: break
                idxs = np.asarray(chunk_idx, int)          # (B,m)
                null = Xc[:, idxs].sum(axis=2)             # (N,B)
                mu, m2, n_seen = update_stream(mu, m2, n_seen, null)
                if abs_variant:
                    null_a = (np.abs(Xc))[:, idxs].sum(axis=2)
                    mu_a, m2_a, n_seen_a = update_stream(mu_a, m2_a, n_seen_a, null_a)
            sd = (np.sqrt(m2 / max(n_seen, 1)) + 1e-6) if n_seen > 0 else np.ones(N)
            if abs_variant:
                sd_a = (np.sqrt(m2_a / max(n_seen_a, 1)) + 1e-6) if n_seen_a > 0 else np.ones(N)
        else:
            idxs = np.stack([rng.choice(pool, size=m, replace=False) for _ in range(n_perm)], axis=0)  # (P,m)
            null = Xc[:, idxs].sum(axis=2)         # (N,P)
            mu = null.mean(axis=1); sd = null.std(axis=1) + 1e-6
            if abs_variant:
                null_a = (np.abs(Xc))[:, idxs].sum(axis=2)
                mu_a = null_a.mean(axis=1); sd_a = null_a.std(axis=1) + 1e-6

        z = (obs[:, s] - mu) / sd
        Z[:, s] = z.astype(np.float32); P[:, s] = p_from_Z(z)
        if abs_variant:
            z_a = (obs_abs[:, s] - mu_a) / sd_a
            Z_abs[:, s] = z_a.astype(np.float32); P_abs[:, s] = p_from_Z(z_a)

    Z_dir = (np.sign(obs) * (Z_abs if abs_variant else np.abs(Z))).astype(np.float32)

    Z_df    = pd.DataFrame(Z,     index=adata.obs_names, columns=set_names)
    P_df    = pd.DataFrame(P,     index=adata.obs_names, columns=set_names)
    Zdir_df = pd.DataFrame(Z_dir, index=adata.obs_names, columns=set_names)
    if abs_variant:
        Zabs_df = pd.DataFrame(Z_abs, index=adata.obs_names, columns=set_names)
        Pabs_df = pd.DataFrame(P_abs, index=adata.obs_names, columns=set_names)
        return Z_df, P_df, Zabs_df, Pabs_df, Zdir_df
    else:
        return Z_df, P_df, Zdir_df

# ---------------------------
# AUROC against manual gates (string-heuristic mapping)
# ---------------------------
def heuristic_labels_to_sets(cell_types: pd.Series, set_names: List[str]) -> Dict[str, np.ndarray]:
    ct = cell_types.astype(str).str.lower().fillna("")
    rules = {
        "Tcell_core": ["t ", " t-", "cd4", "cd8", "tcell", "t cell"],
        "Bcell_core": [" b ", " b-", "bcell", "b cell", "cd19", "cd20", "b220"],
        "NK_core":    ["nk", "natural killer"],
        "Myeloid":    ["mono", "myeloid", "granulo", "neutro", "cd14", "cd33"],
        "HSC_like":   ["stem", "hsc", "cd34"],
        "DC_like":    ["dc", "dendritic"],
        "HSPC":       ["stem", "hspc", "kit", "sca-1"],
    }
    Y = {}
    for s in set_names:
        toks = rules.get(s, [])
        mask = np.zeros(len(ct), bool)
        for t in toks:
            mask |= ct.str.contains(t, regex=False)
        Y[s] = mask
    return Y


def umap_with_progress(
    adata,
    use_rep="X_pca",            # or None to use adata.X
    key_out="X_umap",
    n_neighbors=15,
    min_dist=0.5,
    n_components=2,
    metric="euclidean",
    random_state=None,
    densmap=False,
    **kwargs,                   # passes through to umap.UMAP
):
    import umap
    X = adata.obsm[use_rep] if (use_rep and use_rep in adata.obsm) else adata.X
    um = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric,
        random_state=random_state,
        densmap=densmap,
        verbose=True,          # <- this enables the progress bar
        **kwargs,
    )
    emb = um.fit_transform(X)
    adata.obsm[key_out] = emb.astype("float32")
    return um

def fast_umap_with_progress(
    adata,
    use_rep="X_pca",       # use your PCA here
    key_out="X_umap",
    n_neighbors=15,
    min_dist=0.3,
    n_components=2,
    metric="euclidean",
    n_epochs=120,          # fewer epochs = faster
    init="random",         # faster than 'spectral' for large N
    fit_subset=50_000,     # fit on this many cells, transform the rest
    random_state=1,
    verbose=True,
    **kwargs
):
    import numpy as np, umap
    X = adata.obsm[use_rep] if (use_rep and use_rep in adata.obsm) else adata.X
    N = X.shape[0]
    rng = np.random.RandomState(random_state)

    if (fit_subset is not None) and (N > fit_subset):
        idx_fit = rng.choice(N, fit_subset, replace=False)
        X_fit = X[idx_fit]
    else:
        idx_fit = None
        X_fit = X

    um = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric,
        n_epochs=n_epochs,
        init=init,
        random_state=random_state,
        low_memory=True,          # reduces RAM spikes
        verbose=verbose,          # progress bar
        **kwargs,
    )

    # Optional: pre-warm numba JIT (tiny run) to avoid big first-call lag
    _ = umap.UMAP(n_neighbors=5, n_epochs=10, random_state=0, verbose=False).fit_transform(X[:500])

    emb_fit = um.fit_transform(X_fit)  # progress bar appears here

    if idx_fit is None:
        adata.obsm[key_out] = emb_fit.astype("float32")
    else:
        # Fast interpolation (“transform”) for the rest of the cells
        emb_all = um.transform(X)
        adata.obsm[key_out] = emb_all.astype("float32")
        print(adata.obsm)
    return um




def plot_set_on_umap(adata, scores_df, set_name, label=None, s=8, cmap="viridis",vmin=None,vmax=None):
    if set_name not in scores_df.columns:
        raise KeyError(f"{set_name} not found in provided scores_df columns.")
    key = (label or f"score_{set_name}")
    adata.obs[key] = scores_df[set_name].to_numpy()
    sc.pl.umap(adata, color=[key], s=s, color_map=cmap,vmin=vmin,vmax=vmax)
    return key


In [ ]:
import scanpy as sc

In [ ]:
marker_sets_bc = {
    # ---- Epithelial / luminal / basal identity ----
    "Epithelial_core": {"E-cadherin", "EpCAM", "Pan-KRT", "KRT8-18"},
    "Luminal_ER": {"ER", "GATA3", "KRT8-18", "EpCAM", "E-cadherin"},
    "Luminal_progenitor": {"EpCAM", "CD49f", "CD24", "KRT8-18"},
    "Basal_like": {"KRT5", "CD44", "Vimentin"},              # TNBC-leaning/basal program
    "Myoepithelial": {"aSMA", "KRT5", "CD49f"},
    "Mesenchymal_EMT": {"Vimentin", "aSMA"},                  # pair with Epithelial_core (low) to gauge EMT
    "Stem_CSC_like": {"CD44", "CD49f", "BMI1", "EZH2"},       # CD44hi/CD24lo phenotype (note: CD24 low not encoded here)
    "Basal_Noa":{'H3K4me1', 'H3K9me2', 'H3K4me3'},
    # ---- Chromatin / epigenetic programs ----
    "Active_promoter": {"H3K4me3", "H3K9ac"},
    "Active_enhancer": {"H3K27ac", "H3K4me1"},
    "Transcription_elongation": {"H3K36me3"},
    "Polycomb_repression": {"H3K27me3", "H2AK119ub", "EZH2", "H3K27me2"},
    "Constitutive_heterochromatin": {"H3K9me3", "H3K9me2", "H4K20me3"},
    "Global_acetylation": {"H3K9ac", "H3K27ac", "H3K64ac", "H4K16ac"},

    # ---- Proliferation / stress ----
    "Proliferation": {"KI67", "H3S28p"},
    "DNA_damage_stress": {"pH2A.X", "H3S28p"},

    # ---- Hematopoietic/lineage controls (useful for tumor microenvironment) ----
    "Hematopoietic_markers": {"CD45", "CD47"},               # broad immune features (if present in sample)
    "Myeloid_markers": {"CD14", "CD33", "CD11b", "CD64", "HLA-DR"},
    "Bcell_markers": {"CD19", "CD20", "CD22"},
    "Tcell_markers": {"CD3", "CD4", "CD8", "CD7"},
    "Dendritic_like": {"CD11c", "CD123", "HLA-DR"},

    # ---- Misc useful modules ----
    "Adhesion_migration": {"CD44", "CD49f", "CD24"},
    "Epithelial_polarity": {"E-cadherin", "EpCAM"},
}
# (Optional) prune to your panel defensively:
present = set(['BMI1','CD24','CD44','CD49f','E-cadherin','ER','EZH2','EpCAM','GATA3','H2AK119ub',
               'H3K27ac','H3K27me2','H3K27me3','H3K36me2','H3K36me3','H3K4me1','H3K4me3','H3K64ac',
               'H3K9ac','H3K9me2','H3K9me3','H3S28p','H4K16ac','H4K20me3','KI67','KRT5','KRT8-18',
               'Pan-KRT','Vimentin','aSMA','pH2A.X'])
marker_sets_bc = {k: {m for m in v if m in present} for k, v in marker_sets_bc.items() if any(m in present for m in v)}
marker_sets=marker_sets_bc

In [ ]:
GS=list(marker_sets.keys())
GS.sort()
GS

In [ ]:
try:
    MRK_All.remove('class')
except:
    pass

In [ ]:
def OtherScores(DB,scale=True):
    import scanpy as sc
    import matplotlib.pyplot as plt
    DF=globals()[DB].copy()#sample(5000,replace=False).copy()
    
    UM=umap.UMAP(min_dist=0.01,n_neighbors=15,verbose=True)
    X=UM.fit_transform(DF[MRK_All])    
    DF['class']=globals()[f"Label_{DB}"]
    features = MRK_All
    
    metadata = ["Samp","class"]
    COLS=['Perm_Epithelial_core', 'Perm_Luminal_ER',
        'Perm_Basal_like', 
       'Perm_Mesenchymal_EMT', 'Perm_Basal_Noa','Perm_Proliferation']
    adata = ad.AnnData(DF[features].values, obs=DF[metadata].copy())
    adata.var_names = features
    #adata.obs['Line']=adata.obs['Line'].astype("category")
    adata.obsm["X_umap"] = X
    for col in adata.obs.columns:
        adata.obs[col]=adata.obs[col].astype("category")
    if scale:
        sc.pp.scale(adata)
    sc.pl.umap(adata,color=MRK_All+['Samp','class'],cmap='seismic',vmin='p1',vmax='p99',show=False);
    plt.savefig(f"Plots2/{DB}_UMAP.png",dpi=200,bbox_inches='tight')

    if "cluster" in adata.obs.columns:
        adata.obs.drop(columns=["cluster"], inplace=True)

    # out=ucell_scores(adata,marker_sets)
    # adata.obsm['UCell']=out
    # out=aucell_scores(adata,marker_sets)
    # adata.obsm['AUCell']=out
    out=sipsic_like_scores_v2(adata,marker_sets,abs_variant=True,progress=True)
    adata.obsm['SS_Z']=out[0]
    adata.obsm['SS_P']=out[1]
    adata.obsm['SS_Z_Abs']=out[2]
    adata.obsm['SS_P_Abs']=out[3]
    adata.obsm['SS_Z_dir']=out[4]
    for C in adata.obsm['SS_Z'].columns:
        adata.obs[f'Perm_{C}']=adata.obsm['SS_Z'][C]
    #print(adata.obs)
    
    List=[f'Perm_{g}' for g in GS]
    sc.pl.umap(adata, color=COLS, cmap="seismic",vcenter=0,show=False,)
    plt.savefig(f"Plots2/{DB}_UMAP_MarkerSets_Perm.png",dpi=200,bbox_inches='tight')    

    for C in adata.obsm['SS_Z'].columns:
        adata.obs[f'Perm_{C}_Abs']=adata.obsm['SS_Z_Abs'][C]
    List=[f'Perm_{g}_Abs' for g in GS]
    sc.pl.umap(adata, color=List, cmap="seismic",vcenter=0,show=False,)
    plt.savefig(f"Plots2/{DB}_UMAP_MarkerSets_Perm_Abs.png",dpi=200,bbox_inches='tight')

    for C in adata.obsm['SS_Z'].columns:
        adata.obs[f'Perm_{C}_dir']=adata.obsm['SS_Z_dir'][C]
    List=[f'Perm_{g}_dir' for g in GS]
    sc.pl.umap(adata, 
               color=COLS, cmap="seismic",vcenter=0,show=False,)
    plt.savefig(f"Plots2/{DB}_UMAP_MarkerSets_Perm_dir.png",dpi=200,bbox_inches='tight')


    ### Highlight Top and Bottom Q cells
    import os, numpy as np, pandas as pd
    import scanpy as sc
    import matplotlib.pyplot as plt
    
    # --- params ---
    Q = 0.10                 # tail fraction
    dot_size_other = 4       # base small dots
    dot_size_high  = 8      # larger highlighted dots
    os.makedirs("Plots2", exist_ok=True)
    
    # score columns in adata.obs
    List = [f"Perm_{g}" for g in GS]
    
    N = adata.n_obs
    k = max(1, int(np.ceil(Q * N)))
    
    # build three-level categorical per panel + remember indices per panel
    highlight_cols = []
    tail_indices   = []   # list of dicts per col: {"bottom": idx_array, "top": idx_array}
    
    for col in List:
        if col not in adata.obs:
            raise KeyError(f"{col} not found in adata.obs.")
    
        vals = np.asarray(adata.obs[col].values, dtype=float)
        order = np.argsort(vals)
        bottom_idx = order[:k]
        top_idx    = order[-k:]
    
        labels = np.full(N, "other", dtype=object)
        labels[bottom_idx] = "bottom"
        labels[top_idx]    = "top"
    
        new_key = f"HILIGHT_{col}"
        adata.obs[new_key] = pd.Categorical(
            labels, categories=["other", "bottom", "top"], ordered=True
        )
        highlight_cols.append(new_key)
        tail_indices.append({"bottom": bottom_idx, "top": top_idx})
    
    # --- base plot: all panels with small dots (others in gray, tails colored but same size) ---
    fig = sc.pl.umap(
        adata,
        color=highlight_cols,
        palette={"other": "#d0d0d0", "bottom": "#1f77b4", "top": "#d62728"},
        edges=False,
        s=dot_size_other,       # small for everyone in the base layer
        na_color="#d0d0d0",
        legend_loc=None,        # avoid per-panel legends so axes indexing is clean
        show=False,
        return_fig=True,
    )
    
    # --- overlay: bigger tail points on their respective panels ---
    X = adata.obsm["X_umap"]
    # keep only axes that have data (exclude any accidental cbar axes)
    axes = [ax for ax in fig.axes if ax.has_data()]
    
    for ax, tails in zip(axes, tail_indices):
        bi = tails["bottom"]; ti = tails["top"]
        # bottom Q in blue, bigger
        ax.scatter(X[bi, 0], X[bi, 1], s=dot_size_high, c="#1f77b4",
                   linewidths=0, rasterized=True, zorder=10)
        # top Q in red, bigger
        ax.scatter(X[ti, 0], X[ti, 1], s=dot_size_high, c="#d62728",
                   linewidths=0, rasterized=True, zorder=11)
    
    # optional: a single global legend (outside panels)
    handles = [
        plt.Line2D([], [], marker='o', linestyle='', color="#d0d0d0", label='other', markersize=6),
        plt.Line2D([], [], marker='o', linestyle='', color="#1f77b4", label=f'bottom {int(Q*100)}%', markersize=8),
        plt.Line2D([], [], marker='o', linestyle='', color="#d62728", label=f'top {int(Q*100)}%', markersize=8),
    ]
    # put legend on the last axis (or comment out if not needed)
    axes[-1].legend(handles=handles, frameon=False, loc='upper right',bbox_to_anchor=(2.,1))
    
    plt.savefig(f"Plots2/{DB}_UMAP_MarkerSets_TopBottom{int(Q*100)}_bigHigh.png",
                dpi=200, bbox_inches="tight")
    plt.close(fig)

    
    return adata

In [ ]:
OtherScores('BCK11')

In [ ]:
Lab